# 08 — Model Evaluation, Threshold Optimization & SHAP Explainability
**Project:** Axlero Solutions — Credit Risk Scoring and Loan Default Analysis  
**Module:** Model Evaluation & Explainability (Module 5 — Sowmya)

## 1. Held-Out Test Set Evaluation (`n = 4,863`)
The held-out test set was never used during imputation, WoE binning, feature scaling, feature selection, or hyperparameter tuning.

In [1]:
import sys
import json
from pathlib import Path
import joblib
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.models.train_model import predict_single_borrower

with open(PROJECT_ROOT / "models" / "all_models_summary.json", "r", encoding="utf-8") as f:
    summary_data = json.load(f)
with open(PROJECT_ROOT / "models" / "threshold_analysis.json", "r", encoding="utf-8") as f:
    thresh_data = json.load(f)

comp_df = pd.DataFrame(summary_data["model_comparison"])
print(f"Champion Model: {summary_data['champion_model_name']}")
display(comp_df[[
    "model", "test_accuracy", "test_precision", "test_recall",
    "test_specificity", "test_f1", "test_roc_auc", "test_pr_auc",
    "test_ks_stat", "test_brier", "test_tn", "test_fp", "test_fn", "test_tp"
]])

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Champion Model: Gradient Boosting (Tuned)


,model,test_accuracy,test_precision,test_recall,test_specificity,test_f1,test_roc_auc,test_pr_auc,test_ks_stat,test_brier,test_tn,test_fp,test_fn,test_tp
0,Gradient Boosting (Tuned),0.9439,0.9759,0.7622,0.9947,0.8559,0.9553,0.9199,0.7891,0.0466,3779,20,253,811
1,XGBoost (Champion Tuned),0.9422,0.9769,0.7538,0.9950,0.8509,0.9547,0.9182,0.7818,0.0473,3780,19,262,802
2,XGBoost (Balanced),0.9350,0.8864,0.8064,0.9710,0.8445,0.9529,0.9165,0.7834,0.0576,3689,110,206,858
3,Random Forest (Tuned),0.9297,0.9011,0.7622,0.9766,0.8259,0.9355,0.8946,0.7549,0.0661,3710,89,253,811
4,Decision Tree (Tuned),0.9321,0.9520,0.7265,0.9897,0.8241,0.9154,0.8626,0.7184,0.0591,3760,39,291,773
5,Logistic Regression (Balanced),0.8468,0.6129,0.8139,0.8560,0.6992,0.9017,0.7834,0.6895,0.1173,3252,547,198,866
6,Logistic Regression (L2),0.8888,0.8030,0.6513,0.9553,0.7193,0.8997,0.7983,0.6920,0.0856,3629,170,371,693
7,Dummy Baseline (Stratified),0.6572,0.2153,0.2143,0.7813,0.2148,0.4978,0.2180,0.0000,0.3428,2968,831,836,228


## 2. Decision Threshold Optimization (Default `0.50` vs F1-Optimal `0.39` vs Cost-Optimal `0.16`)
In consumer lending, a False Negative (approving a borrower who defaults, losing principal) is substantially costlier than a False Positive (declining a creditworthy applicant, foregoing interest margin). We compare three operating regimes on the test set:

In [2]:
thresh_summary = pd.DataFrame([
    {"Regime": "Standard Threshold (0.50)", **thresh_data["default_threshold_0_50_test_metrics"]},
    {"Regime": f"F1-Optimal Threshold ({thresh_data['f1_optimal_threshold']})", **thresh_data["f1_optimal_test_metrics"]},
    {"Regime": f"5:1 Cost-Optimal Threshold ({thresh_data['cost_optimal_threshold_5_to_1']})", **thresh_data["cost_optimal_test_metrics"]},
])
display(thresh_summary)

,Regime,threshold,accuracy,precision,recall,specificity,f1_score,roc_auc,pr_auc,gini_coefficient,ks_statistic,brier_score,tn,fp,fn,tp
0,Standard Threshold (0.50),0.50,0.9439,0.9759,0.7622,0.9947,0.8559,0.9553,0.9199,0.9106,0.7891,0.0466,3779,20,253,811
1,F1-Optimal Threshold (0.39),0.39,0.9441,0.9500,0.7857,0.9884,0.8601,0.9553,0.9199,0.9106,0.7891,0.0466,3755,44,228,836
2,5:1 Cost-Optimal Threshold (0.16),0.16,0.9011,0.7315,0.8656,0.9110,0.7929,0.9553,0.9199,0.9106,0.7891,0.0466,3461,338,143,921


## 3. Global & Local SHAP Explainability

In [3]:
shap_df = pd.DataFrame(summary_data["shap_global_importance"])
print("Top 15 Global Features by Mean |SHAP Value|:")
display(shap_df.head(15))

bundle = joblib.load(PROJECT_ROOT / "models" / "credit_risk_pipeline.joblib")
sample_applicant = {
    "person_age": 24,
    "person_income": 32000,
    "person_home_ownership": "RENT",
    "person_emp_length": 1.0,
    "loan_intent": "DEBTCONSOLIDATION",
    "loan_grade": "D",
    "loan_amnt": 12000,
    "loan_int_rate": 15.31,
    "loan_percent_income": 0.38,
    "cb_person_default_on_file": "Y",
    "cb_person_cred_hist_length": 3,
}
pred_out = predict_single_borrower(bundle, sample_applicant)
print(f"\nSingle Applicant Prediction: PD = {pred_out['default_probability_pct']}% | {pred_out['risk_category']} | {pred_out['underwriting_recommendation']}")
display(pd.DataFrame(pred_out["top_factors"]))

Top 15 Global Features by Mean |SHAP Value|:


,feature,mean_abs_shap
0,home_RENT,0.7886
1,log_person_income,0.5877
2,calc_loan_to_income,0.5757
3,home_MORTGAGE,0.3607
4,loan_int_rate,0.3318
5,intent_HOMEIMPROVEMENT,0.2869
6,grade_x_burden,0.2792
7,subprime_grade_flag,0.2650
8,loan_grade_num,0.2425
9,intent_DEBTCONSOLIDATION,0.2341



Single Applicant Prediction: PD = 99.89% | High Risk | Decline or Require Co-Signer & Principal Reduction


,feature,scaled_value,shap_value,abs_shap,direction
0,home_RENT,1.000000,1.774369,1.774369,Increases Default Risk (+)
1,calc_loan_to_income,1.904442,1.701577,1.701577,Increases Default Risk (+)
2,intent_DEBTCONSOLIDATION,1.000000,1.164635,1.164635,Increases Default Risk (+)
3,subprime_grade_flag,1.000000,1.004282,1.004282,Increases Default Risk (+)
4,grade_x_burden,3.096928,0.975561,0.975561,Increases Default Risk (+)
5,loan_grade_num,1.527630,0.801375,0.801375,Increases Default Risk (+)
6,debt_service_to_income,2.018365,0.555537,0.555537,Increases Default Risk (+)
7,loan_int_rate,1.338292,0.405848,0.405848,Increases Default Risk (+)
